# Customer Churn Analysis and Prediction

## A modern, leakage-aware workflow

This notebook moves from business framing and exploratory analysis to model validation, decision-threshold selection, calibration, lift analysis, and interpretable retention insights.

## 1. Business framing

**Customer churn** means the end of a customer relationship. Two separate dimensions are often confused:

- **Contractual vs. non-contractual:** cancellation is observed directly in contractual products; in non-contractual settings it must be inferred from inactivity.
- **Voluntary vs. involuntary:** a customer may choose to leave, or service may end because of payment failure, fraud controls, or another operational event.

A useful churn model needs an explicit **observation window**, **prediction date**, **prediction horizon**, and **action**. Every feature must be available at the prediction date. Otherwise, the model may leak information from the future.

In production, monitor label definitions, feature drift, probability calibration, campaign capacity, and outcomes after intervention—not just model discrimination.

### Learning goals

By the end, you will be able to:

1. audit a churn label and its feature timing;
2. explore churn rates without reversing conditional percentages;
3. compare models with stratified cross-validation and imbalance-aware metrics;
4. evaluate one selected model on an untouched test set;
5. tune an operating threshold using out-of-fold predictions; and
6. translate predictions into calibration, lift, and feature-impact diagnostics.

**Data:** the original [Kaggle telecom churn dataset](https://www.kaggle.com/datasets/barun2104/telecom-churn). A public raw mirror is used only as a fallback when no local CSV exists.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython import get_ipython
from IPython.display import display

import sklearn
from sklearn.base import clone
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

get_ipython().run_line_magic("matplotlib", "inline")
RANDOM_STATE = 42
N_JOBS = 1  # Stable across Windows/Jupyter; this dataset is small.
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

print(f"pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

In [ ]:
LOCAL_PATHS = [Path("data/telecom_churn.csv"), Path("telecom_churn.csv")]
DATA_URL = (
    "https://raw.githubusercontent.com/prasertcbs/"
    "basic-dataset/master/telecom_churn.csv"
)

local_path = next((path for path in LOCAL_PATHS if path.exists()), None)
data_source = local_path if local_path is not None else DATA_URL

data = pd.read_csv(data_source)
print(f"Loaded {data.shape[0]:,} rows × {data.shape[1]} columns from {data_source}")
data.head()

### Data dictionary

- **Churn:** 1 if the customer cancelled service; otherwise 0
- **AccountWeeks:** account tenure in weeks
- **ContractRenewal:** 1 if the customer recently renewed; otherwise 0
- **DataPlan:** 1 if the customer has a data plan; otherwise 0
- **DataUsage:** monthly data usage in GB
- **CustServCalls:** number of customer-service calls
- **DayMins / DayCalls:** average daytime usage and calls
- **MonthlyCharge:** average monthly bill
- **OverageFee:** largest overage fee in the previous 12 months
- **RoamMins:** average roaming minutes

> **Feature-timing warning:** the source does not specify exact timestamps. Before deployment, verify that `ContractRenewal`, `MonthlyCharge`, and `OverageFee` are known before the prediction date and do not encode the churn outcome.

In [ ]:
expected_columns = {
    "Churn", "AccountWeeks", "ContractRenewal", "DataPlan", "DataUsage",
    "CustServCalls", "DayMins", "DayCalls", "MonthlyCharge",
    "OverageFee", "RoamMins",
}
missing_columns = expected_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

print("Data types and non-null counts:")
data.info()

display(pd.DataFrame({
    "missing": data.isna().sum(),
    "unique": data.nunique(),
}).T)
print(f"Duplicate rows: {data.duplicated().sum():,}")

In [ ]:
binary_columns = ["Churn", "ContractRenewal", "DataPlan"]
for column in binary_columns:
    observed = set(data[column].dropna().unique())
    if not observed.issubset({0, 1}):
        raise ValueError(f"{column} must be binary; found {sorted(observed)}")
    data[column] = data[column].astype("int8")

# Keep binary indicators numeric. Casting them to object breaks correlation
# analysis in modern pandas and adds unnecessary one-hot encoded columns.
assert data[binary_columns].isna().sum().sum() == 0
data.dtypes

## 2. Exploratory data analysis

EDA should describe associations, not claim causation. All percentages below are conditioned on the feature value—i.e., “among customers with this value, what fraction churned?”

Each row represents one customer, and `Churn` is the historical outcome. Real systems often need to construct this label from events. That construction must be fixed before modeling: for example, “no paid renewal within 30 days after contract expiry.” Changing the label definition changes the prediction problem.

In [ ]:
data.describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

In [ ]:
target_counts = data["Churn"].value_counts().sort_index()
churn_rate = data["Churn"].mean()

ax = sns.countplot(data=data, x="Churn", hue="Churn", palette="Set2", legend=False)
ax.set(
    title=f"Target balance — churn rate: {churn_rate:.1%}",
    xlabel="Churn (0 = retained, 1 = churned)",
    ylabel="Customers",
)
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}")
plt.show()

target_counts.rename(index={0: "Retained", 1: "Churned"}).to_frame("customers")

In [ ]:
def plot_distribution_by_churn(frame, column, bins=25):
    """Compare a numeric feature's distribution across churn classes."""
    ax = sns.histplot(
        data=frame,
        x=column,
        hue="Churn",
        bins=bins,
        stat="density",
        common_norm=False,
        element="step",
        palette="Set2",
    )
    ax.set_title(f"{column} distribution by churn outcome")
    plt.show()


def churn_rate_by(frame, column):
    """Return P(churn | feature value), not P(feature value | churn)."""
    rates = (
        frame.groupby(column, observed=True)["Churn"]
        .agg(churn_rate="mean", customers="size")
        .reset_index()
    )
    return rates

The target is imbalanced (about 14.5% churn in this dataset). Imbalance is not automatically a data defect, and the test set should retain the real prevalence. We will use stratification, imbalance-aware metrics, class weighting, and threshold selection rather than duplicating observations before splitting.

In [ ]:
continuous_features = [
    "AccountWeeks", "DataUsage", "CustServCalls", "DayMins",
    "DayCalls", "MonthlyCharge", "OverageFee", "RoamMins",
]

axes = data[continuous_features].hist(bins=25, figsize=(14, 9), layout=(3, 3))
for ax in axes.ravel():
    ax.set_ylabel("Customers")
plt.suptitle("Numeric feature distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plot_distribution_by_churn(data, "DataUsage")

In [ ]:
service_call_rates = churn_rate_by(data, "CustServCalls")
ax = sns.barplot(data=service_call_rates, x="CustServCalls", y="churn_rate", color="#4C78A8")
ax.set(title="Churn rises sharply after repeated service calls", ylabel="Observed churn rate")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.show()
service_call_rates.style.format({"churn_rate": "{:.1%}"})

In [ ]:
data_eda = data.copy()
data_eda["DayMinsBand"] = pd.qcut(data_eda["DayMins"], q=5, duplicates="drop")
day_minutes_rates = churn_rate_by(data_eda, "DayMinsBand")

ax = sns.barplot(data=day_minutes_rates, x="DayMinsBand", y="churn_rate", color="#F58518")
ax.set(title="Churn rate by daytime-minutes quintile", xlabel="DayMins quintile", ylabel="Observed churn rate")
ax.tick_params(axis="x", rotation=25)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.show()

In [ ]:
plot_distribution_by_churn(data, "DayCalls")

In [ ]:
data_eda["MonthlyChargeBand"] = pd.qcut(data_eda["MonthlyCharge"], q=5, duplicates="drop")
charge_rates = churn_rate_by(data_eda, "MonthlyChargeBand")

ax = sns.barplot(data=charge_rates, x="MonthlyChargeBand", y="churn_rate", color="#54A24B")
ax.set(title="Churn rate by monthly-charge quintile", xlabel="MonthlyCharge quintile", ylabel="Observed churn rate")
ax.tick_params(axis="x", rotation=25)
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.show()

In [ ]:
plot_distribution_by_churn(data, "OverageFee")

In [ ]:
plot_distribution_by_churn(data, "RoamMins")

### Relationships with churn

Compare groups with conditional churn rates and distributions. Large samples can make tiny differences look important, while small groups can create noisy rates; always inspect group sizes alongside percentages.

These plots identify predictive associations—not causes. For example, repeated service calls may indicate unresolved problems, but intervening on call count itself would not necessarily reduce churn. Causal claims require a designed experiment or defensible causal analysis.

In [ ]:
selected_features = ["ContractRenewal", "CustServCalls", "DayMins", "MonthlyCharge"]
long_data = data.melt(
    id_vars="Churn",
    value_vars=selected_features,
    var_name="feature",
    value_name="value",
)

grid = sns.catplot(
    data=long_data,
    x="Churn",
    y="value",
    col="feature",
    col_wrap=2,
    kind="box",
    sharey=False,
    height=3.2,
    aspect=1.35,
    palette="Set2",
    hue="Churn",
    legend=False,
)
grid.set_axis_labels("Churn", "Value").set_titles("{col_name}")
grid.figure.suptitle("Selected feature distributions by churn", y=1.02)
plt.show()

In [ ]:
correlations = data.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
ax = sns.heatmap(
    correlations,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
)
ax.set(title="Pearson correlation heatmap")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

correlations["Churn"].drop("Churn").sort_values(key=abs, ascending=False).to_frame("correlation_with_churn")

The strongest linear associations with churn include contract renewal, customer-service calls, and daytime minutes. `MonthlyCharge` is strongly related to `DataUsage` and moderately related to `DayMins`; it is **not** strongly related to `DayCalls`. Correlation still misses nonlinear effects and interactions, so it is a screening tool rather than a model-selection rule.

In [ ]:
data_plan_rates = churn_rate_by(data, "DataPlan")
ax = sns.barplot(data=data_plan_rates, x="DataPlan", y="churn_rate", color="#4C78A8")
ax.set(title="Observed churn rate by data-plan status", ylabel="Churn rate")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.show()
data_plan_rates.style.format({"churn_rate": "{:.1%}"})

In [ ]:
renewal_rates = churn_rate_by(data, "ContractRenewal")
ax = sns.barplot(data=renewal_rates, x="ContractRenewal", y="churn_rate", color="#F58518")
ax.set(title="Observed churn rate by renewal status", ylabel="Churn rate")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
plt.show()
renewal_rates.style.format({"churn_rate": "{:.1%}"})

Customer-service calls show one of the clearest nonlinear associations: churn rises sharply after several calls. This corrects the earlier notebook’s conclusion, which normalized its crosstab in the reverse direction.

In [ ]:
pd.crosstab(
    index=data["CustServCalls"],
    columns=data["Churn"],
    normalize="index",
).rename(columns={0: "retained_rate", 1: "churn_rate"}).style.format("{:.1%}")

## 3. Leakage-safe modeling

The workflow below follows four rules:

1. split once and preserve natural class prevalence in the test set;
2. put preprocessing inside each model pipeline;
3. compare candidates only with stratified cross-validation on training data; and
4. evaluate the selected model once on the untouched test set.

In [ ]:
target_summary = data["Churn"].value_counts().sort_index().to_frame("customers")
target_summary["share"] = target_summary["customers"] / len(data)
target_summary.rename(index={0: "Retained", 1: "Churned"}).style.format({"share": "{:.1%}"})

Accuracy alone is misleading when most customers stay: a model that always predicts “retained” is already highly accurate. We therefore prioritize **average precision (PR-AUC)** for ranking rare churners, and also report ROC-AUC, balanced accuracy, precision, recall, F1, calibration, and lift. Metric choice should ultimately reflect contact cost, intervention capacity, and the value of a retained customer.

### Evaluation design

A random row split is acceptable for this timeless teaching dataset. In a real churn system, prefer an **out-of-time split** (train on earlier prediction dates and test on later ones), and group by customer if a customer can appear more than once.

We reserve 25% of rows for final evaluation. `stratify=y` keeps churn prevalence comparable between train and test. The test set will not be resampled or used to select a model.

In [ ]:
target_column = "Churn"
binary_features = ["ContractRenewal", "DataPlan"]
numeric_features = [column for column in data.columns if column not in binary_features + [target_column]]
feature_columns = numeric_features + binary_features

X = data[feature_columns].copy()
y = data[target_column].astype(int)

print(f"Features ({len(feature_columns)}): {feature_columns}")
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "churn_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"])
split_summary.style.format({"churn_rate": "{:.1%}"})

In [ ]:
assert set(X_train.index).isdisjoint(X_test.index)
assert abs(y_train.mean() - y_test.mean()) < 0.01
print("Train/test indices are disjoint and class prevalence is preserved.")

### Candidate models

- **Dummy classifier:** a required prevalence baseline.
- **Logistic regression:** interpretable linear baseline with balanced class weights.
- **Random forest:** nonlinear interactions with balanced class weights.
- **Histogram gradient boosting:** a strong, efficient nonlinear tabular baseline.

Preprocessing is fitted separately inside every cross-validation fold. No manual over- or under-sampling is needed.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("binary", binary_pipeline, binary_features),
], verbose_feature_names_out=False)

estimators = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Logistic regression": LogisticRegression(
        class_weight="balanced", max_iter=2_000, random_state=RANDOM_STATE
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    ),
    "Histogram gradient boosting": HistGradientBoostingClassifier(
        max_iter=250,
        learning_rate=0.05,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

models = {
    name: Pipeline([("preprocess", clone(preprocessor)), ("model", estimator)])
    for name, estimator in estimators.items()
}
list(models)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

print("Primary selection metric: average precision (PR-AUC)")

In [ ]:
cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=N_JOBS,
        error_score="raise",
    )
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"{metric}_std"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = (
    pd.DataFrame(cv_rows)
    .set_index("model")
    .sort_values("average_precision_mean", ascending=False)
)
cv_results.style.format("{:.3f}")

In [ ]:
comparison_metrics = ["average_precision_mean", "roc_auc_mean", "balanced_accuracy_mean"]
plot_data = (
    cv_results[comparison_metrics]
    .rename(columns=lambda name: name.removesuffix("_mean"))
    .reset_index()
    .melt(id_vars="model", var_name="metric", value_name="score")
)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=plot_data, x="score", y="model", hue="metric")
ax.set(title="5-fold cross-validation on training data", xlim=(0, 1), xlabel="Mean validation score")
plt.legend(title=None, loc="lower right")
plt.show()

Cross-validation variability matters: small differences within roughly one standard deviation may not be meaningful. The dummy model also shows why accuracy was omitted from model selection. We select the candidate with the highest mean validation PR-AUC, then fit it on all training rows.

## 4. Final holdout evaluation

The best cross-validated model is now evaluated on the untouched test set at the default 0.50 threshold. This provides an honest estimate for this development cycle; repeated test-driven revisions would turn the test set into another validation set.

Manual downsampling discards useful observations, while upsampling before splitting leaks duplicates into evaluation. If resampling is genuinely needed, apply it only inside training folds with an imbalance-aware pipeline. For this dataset, class weighting plus threshold selection is simpler and preserves the observed data.

In [ ]:
selected_model_name = cv_results.index[0]
final_model = clone(models[selected_model_name])
final_model.fit(X_train, y_train)

test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction_050 = (test_probability >= 0.50).astype(int)

print(f"Selected model: {selected_model_name}")

In [ ]:
def classification_metrics(y_true, probability, threshold=0.50):
    prediction = (probability >= threshold).astype(int)
    return pd.Series({
        "average_precision": average_precision_score(y_true, probability),
        "roc_auc": roc_auc_score(y_true, probability),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "brier_score (lower is better)": brier_score_loss(y_true, probability),
    })

holdout_metrics = classification_metrics(y_test, test_probability)
holdout_metrics.to_frame("test_score").style.format("{:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_prediction_050, ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix at 0.50")
PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[1], name=selected_model_name)
axes[1].set_title("Precision–recall curve")
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[2], name=selected_model_name)
axes[2].set_title("ROC curve")
plt.tight_layout()
plt.show()

In [ ]:
baseline_ap = y_test.mean()
print(f"Test prevalence / random-ranking PR-AUC baseline: {baseline_ap:.3f}")
print(f"Selected model PR-AUC: {holdout_metrics['average_precision']:.3f}")
print(f"Lift over random ranking: {holdout_metrics['average_precision'] / baseline_ap:.2f}×")

PR-AUC should be compared with the churn prevalence, which is the expected score of random ranking. The confusion matrix depends on the threshold; ROC-AUC and PR-AUC evaluate ranking over all thresholds. A good ranking model can still be operationally poor if its probabilities are miscalibrated or its threshold ignores campaign economics.

## 5. Tune the decision threshold

A 0.50 cutoff is a convention, not a business rule. We tune a threshold using **out-of-fold training probabilities**, so the test labels remain untouched. Here, F2 gives recall twice the weight of precision—a reasonable teaching proxy when missing a likely churner is costly.

For a real campaign, replace F2 with explicit expected value: contact cost, offer cost, probability of acceptance, retained margin, and campaign capacity. If only a fixed number of customers can be contacted, selecting the top-scored customers may be clearer than choosing a global threshold.

In [ ]:
oof_probability = cross_val_predict(
    clone(models[selected_model_name]),
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=N_JOBS,
)[:, 1]

oof_precision, oof_recall, thresholds = precision_recall_curve(y_train, oof_probability)
beta = 2
f2_scores = (
    (1 + beta**2) * oof_precision[:-1] * oof_recall[:-1]
    / (beta**2 * oof_precision[:-1] + oof_recall[:-1] + 1e-12)
)
best_index = int(np.argmax(f2_scores))
selected_threshold = float(thresholds[best_index])

print(f"Training OOF F2-optimal threshold: {selected_threshold:.3f}")
print(f"OOF precision: {oof_precision[best_index]:.3f}")
print(f"OOF recall: {oof_recall[best_index]:.3f}")
print(f"OOF F2: {f2_scores[best_index]:.3f}")

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(thresholds, oof_precision[:-1], label="Precision")
plt.plot(thresholds, oof_recall[:-1], label="Recall")
plt.plot(thresholds, f2_scores, label="F2")
plt.axvline(selected_threshold, color="black", linestyle="--", label=f"Selected: {selected_threshold:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Threshold selection from out-of-fold training predictions")
plt.legend()
plt.show()

In [ ]:
threshold_comparison = pd.concat({
    "default_0.50": classification_metrics(y_test, test_probability, threshold=0.50),
    "OOF_F2_tuned": classification_metrics(y_test, test_probability, threshold=selected_threshold),
}, axis=1)

threshold_metrics = ["balanced_accuracy", "precision", "recall", "f1"]
threshold_comparison.loc[threshold_metrics].style.format("{:.3f}")

Threshold tuning changes classification metrics but not ROC-AUC or PR-AUC because it does not change the ranking. Re-estimate the threshold when prevalence, costs, or campaign capacity changes. Do not repeatedly retune against the test set.

## 6. Probability calibration

If a predicted probability will drive expected-value decisions, `0.30` should mean roughly 30% of similar customers churn. Calibration is distinct from ranking quality and should be checked explicitly.

The Brier score measures squared probability error (lower is better). The reliability curve compares predicted risk with observed churn frequency. With a small test set, treat individual bins as noisy.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(
    y_test,
    test_probability,
    n_bins=8,
    strategy="quantile",
    name=selected_model_name,
    ax=ax,
)
ax.set_title("Holdout probability calibration")
plt.show()

print(f"Brier score: {brier_score_loss(y_test, test_probability):.3f}")

## 7. Lift and campaign targeting

Ranking metrics become tangible when customers are sorted by predicted risk. Decile 1 contains the highest-risk 10% of the test population. `cumulative_capture` reports the share of all observed churners found by contacting the highest-risk deciles.

In [ ]:
scored_test = pd.DataFrame({
    "actual_churn": y_test.to_numpy(),
    "churn_probability": test_probability,
})
scored_test["risk_decile"] = (
    pd.qcut(
        scored_test["churn_probability"].rank(method="first", ascending=False),
        q=10,
        labels=False,
    ) + 1
)

lift_table = (
    scored_test.groupby("risk_decile", observed=True)
    .agg(
        customers=("actual_churn", "size"),
        churners=("actual_churn", "sum"),
        average_probability=("churn_probability", "mean"),
    )
    .sort_index()
)
lift_table["churn_rate"] = lift_table["churners"] / lift_table["customers"]
lift_table["lift"] = lift_table["churn_rate"] / y_test.mean()
lift_table["cumulative_capture"] = lift_table["churners"].cumsum() / y_test.sum()
lift_table.style.format({
    "average_probability": "{:.1%}",
    "churn_rate": "{:.1%}",
    "lift": "{:.2f}×",
    "cumulative_capture": "{:.1%}",
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=lift_table.index, y=lift_table["lift"], color="#4C78A8", ax=axes[0])
axes[0].axhline(1, color="black", linestyle="--")
axes[0].set(title="Lift by risk decile", xlabel="Risk decile (1 = highest)", ylabel="Lift")

axes[1].plot(lift_table.index, lift_table["cumulative_capture"], marker="o")
axes[1].plot([1, 10], [0.1, 1.0], linestyle="--", color="gray", label="Random targeting")
axes[1].set(
    title="Cumulative churner capture",
    xlabel="Risk deciles contacted",
    ylabel="Share of churners captured",
    ylim=(0, 1.05),
)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
permutation = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=importance, x="importance_mean", y="feature", color="#E45756")
ax.set(
    title="Holdout permutation importance",
    xlabel="Decrease in average precision when shuffled",
    ylabel=None,
)
plt.show()
importance.style.format({"importance_mean": "{:.3f}", "importance_std": "{:.3f}"})

## 8. Interpretation and production checklist

Permutation importance measures **predictive reliance**, not causal impact. Correlated features can share or mask importance, and a feature useful to the model is not automatically a safe intervention target.

Before deployment:

- create time-aware training and backtesting datasets with explicit prediction dates;
- verify every feature is available before prediction and remove post-outcome leakage;
- compare against the current retention policy with incremental business value;
- calibrate probabilities and choose thresholds from costs/capacity, not accuracy;
- evaluate performance and error rates across relevant customer groups;
- log data versions, model versions, predictions, decisions, and outcomes;
- monitor feature/score drift, churn prevalence, calibration, lift, and realized retention;
- run controlled experiments to estimate whether interventions actually reduce churn; and
- retrain or revisit the label when customer behavior or the product changes.

**Takeaway:** a modern churn system is a decision process, not merely a classifier. Honest validation, calibrated risk, actionable ranking, and measured intervention impact matter more than a single accuracy score.